In [1]:
import os

from datasets import load_dataset
from rdkit import Chem

In [2]:
# ========== Prepare data ==========

data_path = '../pahdb_w24146_all.csv'
saved = '../'
suffix = 'test'  # Options: ['train', 'valid', 'test']
seed = 42

In [3]:
# ========== Split data ==========

dataset = load_dataset('csv', data_files=data_path)
train_test = dataset['train'].train_test_split(test_size=0.2, seed=seed)
valid_test = train_test['test'].train_test_split(test_size=0.5, seed=seed)

train_dataset = train_test['train']
valid_dataset = valid_test['train']
test_dataset = valid_test['test']

print(f"| Train: {len(train_dataset)} | Valid: {len(valid_dataset)} | Test: {len(test_dataset)} |")

| Train: 19316 | Valid: 2415 | Test: 2415 |


In [4]:
# ========== Select data ==========

if suffix == 'train':
    data = train_dataset
elif suffix == 'valid':
    data = valid_dataset
elif suffix == 'test':
    data = test_dataset
else:
    raise ValueError("Suffix must be one of ['train', 'valid', 'test']")

df = data.to_pandas()

In [5]:
def count_heavy_atoms(smiles):
    """Calculates the number of heavy atoms (non-hydrogen) in a SMILES string."""

    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        print(f"Invalid SMILES string: {smiles}")
        return 0
    return sum(1 for atom in mol.GetAtoms() if atom.GetAtomicNum() > 1)

In [6]:
def count_carbon_atoms(smiles):
    """Counts the number of carbon atoms (atomic number 6) in a SMILES string."""

    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        print(f"Invalid SMILES string: {smiles}")
        return 0
    return sum(1 for atom in mol.GetAtoms() if atom.GetAtomicNum() == 6)

In [7]:
# ========== Checks and appends molecular descriptors for chemical filtering ==========

if 'n_heavy' not in df.columns:
    # Heavy atoms are critical for determining the degrees of freedom in vibrations
    print("Warning | Column 'n_heavy' missing. Calculating from 'canonical_smiles'...")
    df['n_heavy'] = df['canonical_smiles'].apply(count_heavy_atoms)
    print("Successfully added 'n_heavy' column.")

if 'n_c' not in df.columns:
    # Carbon count is the primary descriptor for PAH molecular size
    print("\nWarning | Column 'n_c' missing. Calculating from 'canonical_smiles'...")
    df['n_c'] = df['canonical_smiles'].apply(count_carbon_atoms)
    print("Successfully added 'n_c' column.")

Warning | Column 'n_heavy' missing. Calculating from 'canonical_smiles'...
Successfully added 'n_heavy' column.

Warning | Column 'n_c' missing. Calculating from 'canonical_smiles'...
Successfully added 'n_c' column.


In [8]:
# ========== Save data ==========

# Decompose the input data path into directory and file name
folder_name, file_name = os.path.split(data_path)

# Separate the file stem and the extension (e.g., 'zinc_data' and '.csv')
name, ext = os.path.splitext(file_name)

# Construct the modified file name with the specified suffix
new_name = f"{name}_{suffix}{ext}"

# Reassemble the final storage path using the target directory
save_path = os.path.join(saved, new_name)

df.to_csv(save_path, index=False)
print(f"Done! Saved to {save_path}")

Done! Saved to ../pahdb_w24146_all_test.csv
